# Data Preparation: Part 1 - Anomaly Code

## 1. Import Packages & Define Custom Functions

In [1]:
import pandas as pd
import re

In [2]:
# Using custom function from Michael Albert's class

def summarize_dataframe(df):
    missing_values = pd.concat([pd.DataFrame(df.columns, columns=['Variable Name']), 
                      pd.DataFrame(df.dtypes.values.reshape([-1,1]), columns=['Data Type']),
                      pd.DataFrame(df.isnull().sum().values, columns=['Missing Values']), 
                      pd.DataFrame([df[name].nunique() for name in df.columns], columns=['Unique Values'])], 
                     axis=1).set_index('Variable Name')
    return pd.concat([missing_values, df.describe(include='all').transpose()], axis=1).fillna("")

## 2. Import Raw Data

In [3]:
full = pd.read_csv('/home/jovyan/MSBA Mod 5/Final Deliverable/Data Cleaning and Feature Engineering/Raw Input Files/asrs_full.csv')

## 3. Clean Data

### 3.1. Preliminary Category Aggregation

In [4]:
aggregate_others = ['Other uav report', 'Other Fatigue', 'Other UAV', 'Other Service Road signs needed']

full.loc[full['anomaly_code'].isin(aggregate_others), 'anomaly_code'] = 'No Specific Anomaly Occurred All Types'

In [5]:
other_lasers = ['Other LASER']

full.loc[full['anomaly_code'].isin(other_lasers), 'anomaly_code'] = 'Inflight Event / Encounter Laser'

### 3.2. Identify Unique Categories

In [6]:
anomalies = full.anomaly_code

In [7]:
disaggregated_anomalies = []
for phrase in anomalies:
    disaggregated_anomalies.extend([s.strip() for s in re.split(r';', phrase)])

In [8]:
disaggregated_anomalies = pd.Series(disaggregated_anomalies)

In [9]:
# Counts
value_counts = disaggregated_anomalies.value_counts().reset_index()
value_counts.columns = ["Anomaly", "Count"]

In [10]:
with pd.option_context('display.max_rows', None):
    display(value_counts)

,Anomaly,Count
0,Deviation / Discrepancy - Procedural Published...,19131
1,Aircraft Equipment Problem Critical,8227
2,Deviation / Discrepancy - Procedural Clearance,7843
3,ATC Issue All Types,6020
4,Aircraft Equipment Problem Less Severe,5605
5,Deviation / Discrepancy - Procedural FAR,3935
6,Conflict NMAC,3335
7,Inflight Event / Encounter CFTT / CFIT,3180
8,Inflight Event / Encounter Weather / Turbulence,2878
9,Deviation - Track / Heading All Types,2374


### 3.3. Create Dummy Columns

In [11]:
# Obtain dummy columns
dummy_full = full.copy()
dummy_full['anomaly_code'] = dummy_full['anomaly_code'].str.replace(' ', '', regex=False)
dummies = dummy_full['anomaly_code'].str.get_dummies(sep=';')
# dummy_full = pd.concat([full.copy(), dummies], axis=1)

In [12]:
# Rename variables to include prefix "anomaly_"
rename = dummies.columns
rename_to = []
for x in rename:
    rename_to.append("anomaly_" + x)

dummies.columns=rename_to

In [13]:
# Concat with original data
dummy_full = pd.concat([full.acn, full.anomaly_code, dummies], axis=1)

In [14]:
with pd.option_context('display.max_rows', None):
    display(summarize_dataframe(dummy_full))

/tmp/ipykernel_1029/3979652495.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return pd.concat([missing_values, df.describe(include='all').transpose()], axis=1).fillna("")


,Data Type,Missing Values,Unique Values,count,unique,top,freq,mean,std,min,25%,50%,75%,max
acn,int64,0,33723,33723.0,,,,1828811.478783,192110.173434,1507557.0,1673862.5,1806271.0,1981117.5,2296796.0
anomaly_code,object,0,6659,33723.0,6659,Aircraft Equipment Problem Critical,1974,,,,,,,
anomaly_ATCIssueAllTypes,int64,0,2,33723.0,,,,0.178513,0.38295,0.0,0.0,0.0,0.0,1.0
anomaly_AircraftEquipmentProblemCritical,int64,0,2,33723.0,,,,0.243958,0.429474,0.0,0.0,0.0,0.0,1.0
anomaly_AircraftEquipmentProblemLessSevere,int64,0,2,33723.0,,,,0.166207,0.372272,0.0,0.0,0.0,0.0,1.0
anomaly_AirspaceViolationAllTypes,int64,0,2,33723.0,,,,0.037007,0.188783,0.0,0.0,0.0,0.0,1.0
anomaly_ConflictAirborneConflict,int64,0,2,33723.0,,,,0.045103,0.207533,0.0,0.0,0.0,0.0,1.0
anomaly_ConflictGroundConflict,int64,0,2,33723.0,,,,0.068025,0.251792,0.0,0.0,0.0,0.0,1.0
anomaly_ConflictNMAC,int64,0,2,33723.0,,,,0.098894,0.298524,0.0,0.0,0.0,0.0,1.0
anomaly_Critical,int64,0,2,33723.0,,,,0.053495,0.225021,0.0,0.0,0.0,0.0,1.0


In [15]:
# Filter dummy_full to verify that the dummy codes worked as designed

dummy_ID = dummy_full[['acn', 'anomaly_code']]
dummy_cats = dummy_full.iloc[:, -72:]
dummy_cut = pd.concat([dummy_ID, dummy_cats], axis=1)

In [16]:
# View the filtered dataframe to verify

with pd.option_context('display.max_columns', None,
                        'display.max_colwidth', None,
                        'display.width', None):
    display(dummy_cut.head(25))

,acn,anomaly_code,acn,anomaly_code,anomaly_ATCIssueAllTypes,anomaly_AircraftEquipmentProblemCritical,anomaly_AircraftEquipmentProblemLessSevere,anomaly_AirspaceViolationAllTypes,anomaly_ConflictAirborneConflict,anomaly_ConflictGroundConflict,anomaly_ConflictNMAC,anomaly_Critical,anomaly_Deviation-AltitudeCrossingRestrictionNotMet,anomaly_Deviation-AltitudeExcursionFromAssignedAltitude,anomaly_Deviation-AltitudeOvershoot,anomaly_Deviation-AltitudeUndershoot,anomaly_Deviation-SpeedAllTypes,anomaly_Deviation-Track/HeadingAllTypes,anomaly_Deviation/Discrepancy-ProceduralClearance,anomaly_Deviation/Discrepancy-ProceduralFAR,anomaly_Deviation/Discrepancy-ProceduralHazardousMaterialViolation,anomaly_Deviation/Discrepancy-ProceduralLandingWithoutClearance,anomaly_Deviation/Discrepancy-ProceduralMEL/CDL,anomaly_Deviation/Discrepancy-ProceduralMaintenance,anomaly_Deviation/Discrepancy-ProceduralOther/Unknown,anomaly_Deviation/Discrepancy-ProceduralPublishedMaterial/Policy,anomaly_Deviation/Discrepancy-ProceduralSecurity,anomaly_Deviation/Discrepancy-ProceduralUnauthorizedFlightOperations(UAS),anomaly_Deviation/Discrepancy-ProceduralWeightAndBalance,anomaly_FlightDeck/Cabin/AircraftEventIllness/Injury,anomaly_FlightDeck/Cabin/AircraftEventOther/Unknown,anomaly_FlightDeck/Cabin/AircraftEventPassengerElectronicDevice,anomaly_FlightDeck/Cabin/AircraftEventPassengerMisconduct,anomaly_FlightDeck/Cabin/AircraftEventSmoke/Fire/Fumes/Odor,anomaly_GroundEvent/EncounterAircraft,anomaly_GroundEvent/EncounterFOD,anomaly_GroundEvent/EncounterFuelIssue,anomaly_GroundEvent/EncounterGearUpLanding,anomaly_GroundEvent/EncounterGroundEquipmentIssue,anomaly_GroundEvent/EncounterGroundStrike-Aircraft,anomaly_GroundEvent/EncounterJetBlast,anomaly_GroundEvent/EncounterLossOfAircraftControl,anomaly_GroundEvent/EncounterLossOfVLOS(UAS),anomaly_GroundEvent/EncounterObject,anomaly_GroundEvent/EncounterOther/Unknown,anomaly_GroundEvent/EncounterPerson/Animal/Bird,anomaly_GroundEvent/EncounterVehicle,anomaly_GroundEvent/EncounterWeather/Turbulence,anomaly_GroundExcursionRamp,anomaly_GroundExcursionRunway,anomaly_GroundExcursionTaxiway,anomaly_GroundIncursionRamp,anomaly_GroundIncursionRunway,anomaly_GroundIncursionTaxiway,anomaly_InflightEvent/EncounterAircraft,anomaly_InflightEvent/EncounterBird/Animal,anomaly_InflightEvent/EncounterCFTT/CFIT,anomaly_InflightEvent/EncounterFlyAway(UAS),anomaly_InflightEvent/EncounterFuelIssue,anomaly_InflightEvent/EncounterLaser,anomaly_InflightEvent/EncounterLossOfAircraftControl,anomaly_InflightEvent/EncounterObject,anomaly_InflightEvent/EncounterOther/Unknown,anomaly_InflightEvent/EncounterUnstabilizedApproach,anomaly_InflightEvent/EncounterVFRInIMC,anomaly_InflightEvent/EncounterWakeVortexEncounter,anomaly_InflightEvent/EncounterWeather/Turbulence,anomaly_LessSevere,anomaly_NoSpecificAnomalyOccurredAllTypes,anomaly_NoSpecificAnomalyOccurredUnwantedSituation
0,1507557,ATC Issue All Types; Deviation / Discrepancy - Procedural Published Material / Policy,1507557,ATC Issue All Types; Deviation / Discrepancy - Procedural Published Material / Policy,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1513720,Aircraft Equipment Problem Less Severe,1513720,Aircraft Equipment Problem Less Severe,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1513718,Flight Deck / Cabin / Aircraft Event Illness / Injury,1513718,Flight Deck / Cabin / Aircraft Event Illness / Injury,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,1513706,Aircraft Equipment Problem Critical,1513706,Aircraft Equipment Problem Critical,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1513663,Aircraft Equipment Problem Less Severe; Deviation /

In [17]:
# This looks finished, but we may opt to lump all the "other" categories into a single "other" cateogry

## 4. Export Data

In [18]:
# Export
#dummy_full.to_csv('Output/Data 1 - anomaly clean.csv', index=False)